# TP 1.1 — Instructor's complete solution

**IMT · Introduction to Machine Learning · Google Colab**

**Private correction — not for distribution before the exercise.**

This solves TP 1.1 on **two concentric circles**, using the same model, loss, optimizer, seed and dataset as the student notebook, including both optional bonus exercises.

Run cells **top to bottom** in Google Colab. No GPU required.

## Scenario — Classify points on two concentric circles

Two features per point, and one of two classes (0 or 1). A straight line is not an appropriate separator for concentric rings; a small non-linear MLP can represent a curved boundary.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

X, y = make_circles(n_samples=800, noise=0.16, factor=0.48, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
scaler = StandardScaler().fit(X_train)  # Fit on training data only.
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train[:, None], dtype=torch.float32)
xte = torch.tensor(X_test, dtype=torch.float32)
yte = torch.tensor(y_test[:, None], dtype=torch.float32)
train_loader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', s=12)
ax.set(title='Training data: two concentric circles', xlabel='Feature 1', ylabel='Feature 2')
plt.show()
print('Training samples:', len(X_train), '| Test samples:', len(X_test))

**Quick check.** Each point has **2 input features**, and the output has **2 possible classes**.

## 1. Linear baseline

Logistic regression is linear in the input features, so a straight-line decision boundary cannot correctly separate concentric rings.

In [ ]:
baseline = LogisticRegression().fit(X_train, y_train)
baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test))
print(f'Linear baseline test accuracy: {baseline_accuracy:.3f}')

## 2. Solution — the MLP

Architecture **2 → 12 → 12 → 1**, with a **ReLU** after each hidden layer and no final sigmoid. The last layer outputs logits, not probabilities.

In [ ]:
def make_model():
    return nn.Sequential(
        nn.Linear(2, 12),
        nn.ReLU(),
        nn.Linear(12, 12),
        nn.ReLU(),
        nn.Linear(12, 1),
    )

model = make_model()
assert model(xtr[:5]).shape == (5, 1)
print(model)

## 3. Solution — loss and optimizer

**BCEWithLogitsLoss** is designed for binary classification, and combines a numerically stable sigmoid with binary cross-entropy. Adam optimizes the network parameters.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
print(type(criterion).__name__, type(optimizer).__name__)

## 4. Solution — training loop

For each mini-batch: **zero gradients → forward pass → loss → backward pass → parameter update**. We train for **80 epochs**.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
    return total_loss / len(loader.dataset)

history = []
for epoch in range(80):
    history.append(train_one_epoch(model, train_loader, criterion, optimizer))

plt.plot(history)
plt.xlabel('Epoch'); plt.ylabel('Training BCE loss'); plt.title('Learning curve')
plt.show()

## 5. Solution — predictions and confusion matrix

Convert raw **logits → probabilities** with sigmoid. Use a threshold of 0.5 to convert probabilities into 0/1 predictions. Accuracy measures the fraction of correct labels, not the closeness of real-valued outputs.

In [ ]:
model.eval()
with torch.no_grad():
    test_logits = model(xte)
    test_prob = torch.sigmoid(test_logits).numpy().ravel()
    test_pred = (test_prob >= 0.5).astype(int)

assert test_prob.min() >= 0 and test_prob.max() <= 1
assert test_pred.shape == y_test.shape
nn_accuracy = accuracy_score(y_test, test_pred)
print(f'Neural network test accuracy: {nn_accuracy:.3f} (linear: {baseline_accuracy:.3f})')
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, cmap='Blues')
plt.show()

## 6. Decision boundaries (provided plotting code)

The logistic-regression boundary is linear, whereas the neural network learns a non-linear decision boundary that fits the circles.

In [ ]:
def plot_boundary(ax, predict_fn, title):
    grid_x, grid_y = np.meshgrid(
        np.linspace(X_test[:, 0].min()-0.6, X_test[:, 0].max()+0.6, 170),
        np.linspace(X_test[:, 1].min()-0.6, X_test[:, 1].max()+0.6, 170),
    )
    grid = np.c_[grid_x.ravel(), grid_y.ravel()]
    probs = predict_fn(grid).reshape(grid_x.shape)
    ax.contourf(grid_x, grid_y, probs, levels=np.linspace(0, 1, 11), cmap='coolwarm', alpha=.6)
    ax.contour(grid_x, grid_y, probs, levels=[.5], colors='k', linewidths=1)
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='coolwarm', edgecolor='k', linewidth=.3, s=16)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(11,4))
plot_boundary(axes[0], lambda z: baseline.predict_proba(z)[:,1], 'Logistic regression')
def predict_nn(z):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(z, dtype=torch.float32))).numpy().ravel()
plot_boundary(axes[1], predict_nn, 'Neural network')
plt.tight_layout(); plt.show()

## 7. Model answers to the written questions

**Q1. Which model performed better?** The non-linear neural network, usually substantially. Run the evaluation cell for the exact accuracies. The particular numbers depend on the fixed seed and training run.

**Q2. Why are ReLU activations useful for circular data?** A sequence of linear layers with no non-linear activations collapses into a single affine map, hence has only a straight-line decision boundary. ReLU permits piecewise-linear compositions that approximate a curved boundary.

**Q3. Why evaluate on the test set?** We want to measure generalization to unseen examples. Accuracy on training data may be optimistic because model parameters were adjusted using those labels.

## ⭐ BONUS 1 — Vary the classification threshold

The threshold controls the trade-off between false positives and false negatives. Below we calculate accuracy and the number of false positives for 0.3, 0.5 and 0.7.

In [ ]:
for threshold in [0.3, 0.5, 0.7]:
    predictions = (test_prob >= threshold).astype(int)
    accuracy = accuracy_score(y_test, predictions)
    false_positives = np.sum((predictions == 1) & (y_test == 0))
    print(f'Threshold {threshold:.1f} | Accuracy: {accuracy:.3f} | False positives: {false_positives}')

**Interpretation:** A higher threshold generally predicts class 1 less often, which tends to reduce false positives but can increase false negatives. For instance, unnecessary medical follow-up or fraud investigations may make false positives costly; missing serious conditions or real fraud may make false negatives costly.

## ⭐ BONUS 2 — Change the hidden-layer width

Try **4, 12, 32** neurons. This comparison is illustrative: one random seed is not enough to claim a particular width is universally best.

In [ ]:
def accuracy_with_width(width):
    torch.manual_seed(SEED)
    candidate = nn.Sequential(
        nn.Linear(2, width),
        nn.ReLU(),
        nn.Linear(width, width),
        nn.ReLU(),
        nn.Linear(width, 1),
    )
    candidate_optimizer = torch.optim.Adam(candidate.parameters(), lr=0.01)
    for epoch in range(80):
        train_one_epoch(candidate, train_loader, criterion, candidate_optimizer)

    candidate.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(candidate(xte)).numpy().ravel()
    predictions = (probabilities >= 0.5).astype(int)
    return accuracy_score(y_test, predictions)

for width in [4, 12, 32]:
    score = accuracy_with_width(width)
    print(f'{width:2d} neurons per hidden layer -> test accuracy: {score:.3f}')

**Interpretation:** Larger networks can represent more complex boundaries but may overfit, cost more to train or converge differently. Evaluate across multiple seeds/datasets before comparing architectures.

**Final takeaway:** Non-linear networks can learn curved decision boundaries; task-specific losses and held-out evaluation are essential.